In [3]:
import csv
import numpy as np

In [7]:
def load_toy_dataset_csv(file_path):
    X = []
    y = []
    
    with open(file_path, mode='r',encoding='utf-8') as f:
        reader = csv.reader(f)
        # Пропускаем заголовок (header)
        header = next(reader) 
        
        for row in reader:
            # Преобразуем строку в числа с плавающей точкой
            float_row = [float(val) for val in row]
            # Все столбцы кроме последнего — это признаки (X)
            X.append(float_row[:-1])
            # Последний столбец — это метка (y)
            label = float_row[-1]
            # Конвертируем метку в {-1, 1} для Hinge Loss 
            y.append(1.0 if label == 1 else -1.0)
            
    return np.array(X), np.array(y)

In [11]:
X_toy_t, y_toy_t = load_toy_dataset_csv(r'C:\Users\mande\Downloads\Datasets PA2-20260428\toydata_tiny.csv')

In [12]:
def get_folds(X, y, k=5):
    indices = np.arange(len(y))
    np.random.shuffle(indices) # Перемешиваем для честности
    
    # Делим индексы на K групп
    fold_sizes = np.full(k, len(y) // k)
    fold_sizes[:len(y) % k] += 1
    current = 0
    folds = []
    for size in fold_sizes:
        start, stop = current, current + size
        folds.append(indices[start:stop])
        current = stop
    return folds

def cross_validation(model_class, X, y, k=5, **model_params):
    folds = get_folds(X, y, k)
    scores = []
    
    for i in range(k):
        # Тестовый фолд — текущий i
        test_idx = folds[i]
        # Обучающий фолд — все остальные
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        
        # Создаем и обучаем твой рукописный SVM
        model = model_class(**model_params)
        model.train(X_train, y_train)
        
        # Считаем точность (Accuracy) 
        preds = model.predict(X_test)
        accuracy = np.mean(preds == y_test)
        scores.append(accuracy)
        
    return np.mean(scores), np.std(scores)

In [13]:
class LargeScaleSVM:
    def __init__(self, optimizer='sgd', lr=0.01, lambda_reg=0.01, batch_size=32, momentum=0.9):
        self.optimizer = optimizer
        self.lr = lr
        self.lambda_reg = lambda_reg
        self.batch_size = batch_size
        self.momentum_param = momentum
        self.w = None
        self.history = [] # Для графиков сходимости 

    def train(self, X, y, epochs=10):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        
        # Переменные для оптимизаторов
        v = np.zeros(n_features) # Для Momentum
        G = np.zeros(n_features) # Для Adagrad
        eps = 1e-8

        for epoch in range(epochs):
            # Перемешивание данных перед каждой эпохой [cite: 32]
            indices = np.random.permutation(n_samples)
            X, y = X[indices], y[indices]
            
            epoch_loss = 0
            for i in range(0, n_samples, self.batch_size):
                X_batch = X[i:i+self.batch_size]
                y_batch = y[i:i+self.batch_size]
                
                # 1. Считаем градиент (hinge loss + L2) [cite: 70, 78]
                # Формула из задания требует найти градиент для каждого класса
                # Для бинарного случая это упрощается до:
                grad = self.lambda_reg * self.w
                for xi, yi in zip(X_batch, y_batch):
                    if yi * np.dot(self.w, xi) < 1:
                        grad -= (yi * xi) / len(y_batch)
                
                # 2. Обновляем веса в зависимости от выбранного оптимизатора 
                if self.optimizer == 'sgd':
                    self.w -= self.lr * grad
                
                elif self.optimizer == 'momentum':
                    v = self.momentum_param * v + self.lr * grad
                    self.w -= v
                
                elif self.optimizer == 'adagrad':
                    G += grad**2
                    self.w -= (self.lr / (np.sqrt(G) + eps)) * grad
            
            # Сохраняй loss для отчета 
            self.history.append(self.calculate_loss(X, y))

    def predict(self, X):
        return np.sign(np.dot(X, self.w))

    def calculate_loss(self, X, y):
        # Реализуй формулу (1) из задания [cite: 70]
        distances = 1 - y * np.dot(X, self.w)
        hinge_loss = np.mean(np.maximum(0, distances))
        reg_loss = 0.5 * self.lambda_reg * np.dot(self.w, self.w)
        return hinge_loss + reg_loss

In [14]:
for opt in ['sgd', 'momentum', 'adagrad']:
    acc, duration = cross_validate_svm(X_toy_t, y_toy_t, opt, batch_size=32)
    print(f"Result: {opt.upper()} -> Accuracy: {acc:.4f}, Time: {duration:.2f}s")

NameError: name 'cross_validate_svm' is not defined